

# Route Choice

In this example, we show how to perform route choice set generation using BFSLE and Link penalisation, for a city in La
Serena Metropolitan Area in Chile.


## Running on Google Colab

Press here to open this notebook in Google Colab <a href="https://colab.research.google.com/github/outerl/AequilibraE-demo/blob/main/route_choice_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

In [ ]:
# Imports
from uuid import uuid4
from tempfile import gettempdir
from pathlib import Path
import logging
import sys
import numpy as np
import folium
from aequilibrae.paths import RouteChoice
from aequilibrae.utils.create_example import create_example

In [ ]:
# We create the example project inside our temp folder
fldr = Path(gettempdir()) / uuid4().hex

project = create_example(fldr, "coquimbo")

## Model parameters



We'll set the parameters for our route choice model. These are the parameters that will be used to calculate the
utility of each path. In our example, the utility is equal to $distance * theta$,
and the path overlap factor (PSL) is equal to $beta$.



In [3]:
# Distance factor
theta = 0.00011

# PSL parameter
beta = 1.1

Let's select a set of nodes of interest



In [4]:
nodes_of_interest = (71645, 74089, 77011, 79385)

Let's build all graphs



In [5]:
project.network.build_graphs()
# We get warnings that several fields in the project are filled with NaNs.
# This is true, but we won't use those fields.

c:\src\outerloop\AequilibraE-demo\.venv\Lib\site-packages\aequilibrae\paths\graph.py:218: UserWarning: Found centroids not present in the graph!
[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108
 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 131 132 133]
  build_compressed_graph(self, remove_dead_ends)
c:\src\outerloop\AequilibraE-demo\.venv\Lib\site-packages\aequilibrae\paths\graph.py:218: UserWarning: Found centroids not present in the graph!
[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  3

We also see what graphs are available



In [6]:
project.network.graphs.keys()

dict_keys(['b', 'c', 't', 'w'])

We grab the graph for cars



In [7]:
graph = project.network.graphs["c"]

# Let's say that utility is just a function of distance, so we build our 'utility' field as distance * theta
graph.network = graph.network.assign(utility=graph.network.distance * theta)

# Prepare the graph with all nodes of interest as centroids
graph.prepare_graph(np.array(nodes_of_interest))

# And set the cost of the graph the as the utility field just created
graph.set_graph("utility")

## Mock demand matrix
We'll create a mock demand matrix with demand 1 for every zone and prepare it for computation.



In [8]:
from aequilibrae.matrix import AequilibraeMatrix

names_list = ["demand", "5x demand"]

mat = AequilibraeMatrix()
mat.create_empty(zones=graph.num_zones, matrix_names=names_list, memory_only=True)
mat.index = graph.centroids[:]
mat.matrices[:, :, 0] = np.full((graph.num_zones, graph.num_zones), 10.0)
mat.matrices[:, :, 1] = np.full((graph.num_zones, graph.num_zones), 50.0)
mat.computational_view()

## Create plot function
Before dive into the Route Choice class, let's define a function to plot assignment results.



In [9]:
def plot_results(link_loads):

    link_loads = link_loads[link_loads["demand_tot"] > 0]
    max_load = link_loads["demand_tot"].max()
    links = project.network.links.data
    loaded_links = links.merge(link_loads, on="link_id", how="inner")

    # Maximum thickness we would like is probably a 10, so let's make sure we don't go over that
    factor = 10 / max_load

    return loaded_links.explore(
        color="red",
        style_kwds={
            "style_function": lambda x: {
                "weight": x["properties"]["demand_tot"] * factor,
            }
        },
    )

## Route Choice class
Here we'll construct and use the Route Choice class to generate our route sets



This object construct might take a minute depending on the size of the graph due to the construction of the compressed
link to network link mapping that's required. This is a one time operation per graph and is cached.



In [10]:
rc = RouteChoice(graph)

# Let's check the default parameters for the Route Choice class
print(rc.default_parameters)

{'generic': {'seed': 0, 'max_routes': 0, 'max_depth': 0, 'max_misses': 100, 'penalty': 1.01, 'cutoff_prob': 0.0, 'beta': 1.0, 'store_results': True}, 'link-penalisation': {}, 'bfsle': {'penalty': 1.0}}


Let's add the demand. If it's not provided, link loading cannot be performed.



In [11]:
rc.add_demand(mat)

It is highly recommended to set either ``max_routes`` or ``max_depth`` to prevent runaway results.



In [12]:
rc.set_choice_set_generation("bfsle", max_routes=5)

We can now perform a computation for single OD pair if we'd like. Here we do one between the first and last centroid
as well as an assignment.



In [13]:
results = rc.execute_single(77011, 74089, demand=1.0)

Because we asked it to also perform an assignment we can access the various results from that.



In [14]:
res = rc.get_results()
res.head()

,origin id,destination id,cost,mask,path overlap,probability,route set
0,77011,74089,1.886882,1,0.214404,0.170793,"[-24222, 30332, 30333, -10435, 30068, 30069, 1..."
1,77011,74089,1.888902,1,0.221983,0.176474,"[-24222, 30332, 30333, -10435, 30068, 30069, 1..."
2,77011,74089,1.907420,1,0.352280,0.274921,"[-24222, 30332, 30333, -10435, 30068, 30069, 1..."
3,77011,74089,1.891147,1,0.221056,0.175343,"[-24222, 30332, 30333, -10435, 30068, 30069, 1..."
4,77011,74089,1.889515,1,0.254836,0.202468,"[-24222, 30332, 30333, -10435, 30068, 30069, 1..."


In [15]:
plot_results(rc.get_load_results())

## Batch operations
To perform a batch operation we need to prepare the object first. We can either provide a list of tuple of the OD
pairs we'd like to use, or we can provided a 1D list and the generation will be run on all permutations.



In [16]:
rc.prepare()

Now we can perform a batch computation with an assignment



In [17]:
rc.execute(perform_assignment=True)
res = rc.get_results()
res.head()

,origin id,destination id,cost,mask,path overlap,probability,route set
0,71645,74089,0.604384,1,0.235265,0.137852,"[-19550, -19549, -19548, -19547, -19546, -1954..."
1,71645,74089,0.605965,1,0.239852,0.140317,"[-19550, -19549, -19548, -19547, -19546, -1954..."
2,71645,74089,0.611944,1,0.435865,0.253468,"[-19550, -19549, -19548, -19547, -19546, -1954..."
3,71645,74089,0.622079,1,0.535938,0.308521,"[-19550, -19549, -19548, -19547, -19546, -1954..."
4,71645,74089,0.606404,1,0.273345,0.159841,"[-19550, -19549, -19548, -19547, -19546, -1954..."


Since we provided a matrix initially we can also perform link loading based on our assignment results.



In [18]:
rc.get_load_results()

,demand_ab,demand_ba,5x demand_ab,5x demand_ba,demand_tot,5x demand_tot
link_id,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0
12,0.0,0.0,0.0,0.0,0.0,0.0
13,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...
34938,0.0,0.0,0.0,0.0,0.0,0.0
34939,0.0,0.0,0.0,0.0,0.0,0.0
34940,0.0,0.0,0.0,0.0,0.0,0.0


We can plot these as well



In [19]:
plot_results(rc.get_load_results())

## Select link analysis
We can also enable select link analysis by providing the links and the directions that we are interested in. Here we
set the select link to trigger when (7369, 1) and (20983, 1) is utilised in "sl1" and "sl2" when (7369, 1) is
utilised.



In [20]:
rc.set_select_links({"sl1": [[(7369, 1), (20983, 1)]], "sl2": [[(7369, 1)]]})
rc.execute(perform_assignment=True)

We can get then the results in a Pandas DataFrame for both the network.



In [21]:
sl = rc.get_select_link_loading_results()
sl

,demand_sl1_ab,demand_sl1_ba,5x demand_sl1_ab,5x demand_sl1_ba,demand_sl2_ab,demand_sl2_ba,5x demand_sl2_ab,5x demand_sl2_ba,demand_sl1_tot,5x demand_sl1_tot,demand_sl2_tot,5x demand_sl2_tot
link_id,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
34938,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
34939,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
34940,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


We can also access the OD matrices for this link loading. These matrices are sparse and can be converted to
SciPy sparse matrices for ease of use. They're stored in a dictionary where the key is the matrix name concatenated
with the select link set name via an underscore.



In [25]:
sl = rc.get_select_link_od_matrix_results()
sl

{'sl1': {'demand': <aequilibrae.matrix.sparse_matrix.COO at 0x24f29301ba0>,
  '5x demand': <aequilibrae.matrix.sparse_matrix.COO at 0x24f293037c0>},
 'sl2': {'demand': <aequilibrae.matrix.sparse_matrix.COO at 0x24f293026e0>,
  '5x demand': <aequilibrae.matrix.sparse_matrix.COO at 0x24f293005e0>}}

In [26]:
od_matrix = sl["sl1"]["demand"]
od_matrix.to_scipy().toarray()

array([[0.        , 0.        , 0.        , 3.00722155],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ]])

In [24]:
project.close()

INFO:aequilibrae:Closed project on C:\Users\Pedro\AppData\Local\Temp\7db0dc8c05f54564b9ad9b1e63c5dd22
